In [0]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
 
CATALOG            = "clutchlytics"
BRONZE_SCOREBOARD  = f"{CATALOG}.bronze.raw_nhl_scoreboard"
BRONZE_SUMMARIES   = f"{CATALOG}.bronze.raw_nhl_game_summaries"
DIM_GAMES          = f"{CATALOG}.silver.dimGames"
DIM_TEAMS          = f"{CATALOG}.silver.dimTeams"
SILVER_TABLE       = f"{CATALOG}.silver.fctGames"
 
SPORT  = "hockey"
LEAGUE = "nhl"
 
print(f"Scoreboard source : {BRONZE_SCOREBOARD}")
print(f"Summaries source  : {BRONZE_SUMMARIES}")
print(f"dimGames ref      : {DIM_GAMES}")
print(f"Target            : {SILVER_TABLE}")

In [0]:
# ── READ BRONZE SOURCES ───────────────────────────────────────────────────────
 
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timezone
import json
 
scoreboard_df = spark.table(BRONZE_SCOREBOARD)
summaries_df  = spark.table(BRONZE_SUMMARIES)
 
# Completed games only — fctGames is a results table
scoreboard_df = scoreboard_df.filter(F.col("completed") == True)
summaries_df  = summaries_df.filter(
    (F.col("home_score").isNotNull()) &
    (F.col("away_score").isNotNull())
)
 
print(f"Scoreboard rows (completed) : {scoreboard_df.count()}")
print(f"Summaries rows              : {summaries_df.count()}")

In [0]:
# ── LOAD DIM REFERENCES ───────────────────────────────────────────────────────
 
dim_games = (
    spark.table(DIM_GAMES)
    .filter(F.col("league") == LEAGUE)
    .select(
        F.col("clutch_game_id"),
        F.col("source_event_id").alias("dim_event_id"),
        F.col("clutch_home_team_id"),
        F.col("clutch_away_team_id"),
        F.col("game_number_in_series"),
        F.col("series_key"),
    )
)
 
print(f"dimGames rows ({LEAGUE}): {dim_games.count()}")

In [0]:
# ── PARSE TEAM STATS FROM SUMMARIES ──────────────────────────────────────────
# Extract team-level stats from boxscore_json.
# One row per game with home_ and away_ prefixed columns.
 
summaries_rows = summaries_df.collect()
team_stats     = []
 
for row in summaries_rows:
    event_id = row["event_id"]
 
    try:
        boxscore  = json.loads(row["boxscore_json"])
        teams     = boxscore.get("teams", [])
    except Exception as e:
        print(f"  WARNING: Could not parse boxscore for event {event_id}: {e}")
        teams = []
 
    def get_stat(team_block, stat_name):
        """Extract stat by name from team statistics array."""
        for stat in team_block.get("statistics", []):
            if stat.get("name") == stat_name:
                val = stat.get("displayValue")
                try:
                    return float(val) if "." in str(val) else int(val)
                except (TypeError, ValueError):
                    return None
        return None
 
    # Map home and away blocks
    home_block = next((t for t in teams if t.get("homeAway") == "home"), {})
    away_block = next((t for t in teams if t.get("homeAway") == "away"), {})
 
    team_stats.append({
        "event_id":              event_id,
        "attendance":            row["attendance"],
 
        # ── Home team stats ──
        "home_shots":            get_stat(home_block, "shotsTotal"),
        "home_hits":             get_stat(home_block, "hits"),
        "home_blocked_shots":    get_stat(home_block, "blockedShots"),
        "home_takeaways":        get_stat(home_block, "takeaways"),
        "home_giveaways":        get_stat(home_block, "giveaways"),
        "home_pp_goals":         get_stat(home_block, "powerPlayGoals"),
        "home_pp_opportunities": get_stat(home_block, "powerPlayOpportunities"),
        "home_pp_pct":           get_stat(home_block, "powerPlayPct"),
        "home_faceoffs_won":     get_stat(home_block, "faceoffsWon"),
        "home_faceoff_pct":      get_stat(home_block, "faceoffPercent"),
        "home_penalties":        get_stat(home_block, "penalties"),
        "home_penalty_minutes":  get_stat(home_block, "penaltyMinutes"),
        "home_sh_goals":         get_stat(home_block, "shortHandedGoals"),
 
        # ── Away team stats ──
        "away_shots":            get_stat(away_block, "shotsTotal"),
        "away_hits":             get_stat(away_block, "hits"),
        "away_blocked_shots":    get_stat(away_block, "blockedShots"),
        "away_takeaways":        get_stat(away_block, "takeaways"),
        "away_giveaways":        get_stat(away_block, "giveaways"),
        "away_pp_goals":         get_stat(away_block, "powerPlayGoals"),
        "away_pp_opportunities": get_stat(away_block, "powerPlayOpportunities"),
        "away_pp_pct":           get_stat(away_block, "powerPlayPct"),
        "away_faceoffs_won":     get_stat(away_block, "faceoffsWon"),
        "away_faceoff_pct":      get_stat(away_block, "faceoffPercent"),
        "away_penalties":        get_stat(away_block, "penalties"),
        "away_penalty_minutes":  get_stat(away_block, "penaltyMinutes"),
        "away_sh_goals":         get_stat(away_block, "shortHandedGoals"),
    })
 
team_stats_df = spark.createDataFrame(team_stats)
print(f"Team stats rows parsed: {team_stats_df.count()}")
 
# Spot check — print first game team stats
print("\nSample team stats (first game):")
team_stats_df.show(1, truncate=False)

In [0]:
# ── BUILD fctGames FROM SCOREBOARD ───────────────────────────────────────────
 
ingested_at = datetime.now(timezone.utc).isoformat()
 
fct_df = (
    scoreboard_df
    .select(
        # ── Natural keys ──
        F.col("event_id").alias("source_event_id"),
        F.lit(SPORT).alias("sport"),
        F.lit(LEAGUE).alias("league"),
 
        # ── Team abbreviations ──
        F.col("home_team_abbr"),
        F.col("away_team_abbr"),
 
        # ── Result ──
        F.col("home_score").cast("integer").alias("home_score"),
        F.col("away_score").cast("integer").alias("away_score"),
        F.col("home_winner").cast("boolean").alias("home_winner"),
        F.col("away_winner").cast("boolean").alias("away_winner"),
        F.when(F.col("home_ot").isNotNull(), True)
         .otherwise(False)
         .alias("went_to_ot"),
 
        # ── Period scores ──
        F.col("home_p1").cast("integer").alias("home_p1"),
        F.col("home_p2").cast("integer").alias("home_p2"),
        F.col("home_p3").cast("integer").alias("home_p3"),
        F.col("home_ot").cast("integer").alias("home_ot"),
        F.col("away_p1").cast("integer").alias("away_p1"),
        F.col("away_p2").cast("integer").alias("away_p2"),
        F.col("away_p3").cast("integer").alias("away_p3"),
        F.col("away_ot").cast("integer").alias("away_ot"),
 
        # ── Series context ──
        F.col("home_series_wins").cast("integer").alias("home_series_wins"),
        F.col("away_series_wins").cast("integer").alias("away_series_wins"),
        F.when(
            (F.col("home_series_wins").cast("integer") == 4) |
            (F.col("away_series_wins").cast("integer") == 4),
            True
        ).otherwise(False).alias("series_clinched"),
        F.col("series_summary"),
        F.col("round").cast("integer").alias("round"),
 
        # ── Featured athletes ──
        F.col("winning_goalie_id"),
        F.col("winning_goalie_name"),
        F.col("winning_goalie_saves").cast("integer").alias("winning_goalie_saves"),
        F.col("winning_goalie_sv_pct").cast("double").alias("winning_goalie_sv_pct"),
        F.col("losing_goalie_id"),
        F.col("losing_goalie_name"),
        F.col("losing_goalie_saves").cast("integer").alias("losing_goalie_saves"),
        F.col("losing_goalie_sv_pct").cast("double").alias("losing_goalie_sv_pct"),
        F.col("first_star_id"),
        F.col("first_star_name"),
        F.col("second_star_id"),
        F.col("second_star_name"),
        F.col("third_star_id"),
        F.col("third_star_name"),
    )
)
 
# ── Join team stats from summaries ──
fct_df = (
    fct_df
    .join(
        team_stats_df,
        fct_df.source_event_id == team_stats_df.event_id,
        how="left"
    )
    .drop("event_id")
)
 
# ── Join dimGames → clutch_game_id, team FKs, game context ──
fct_df = (
    fct_df
    .join(
        dim_games,
        fct_df.source_event_id == dim_games.dim_event_id,
        how="left"
    )
    .drop("dim_event_id")
)
 
# ── Warn on unmatched dimGames joins ──
unmatched = fct_df.filter(F.col("clutch_game_id").isNull()).count()
print(f"Unmatched dimGames joins: {unmatched} {'✓' if unmatched == 0 else '<-- investigate'}")
 
# ── Warn on unmatched team stats joins ──
no_team_stats = fct_df.filter(F.col("home_shots").isNull()).count()
print(f"Games missing team stats: {no_team_stats} {'✓' if no_team_stats == 0 else '<-- investigate'}")
 
# ── Final column order ──
fct_df = fct_df.select(
    # ── Keys ──
    "clutch_game_id",
    "source_event_id",
    "sport",
    "league",
    "clutch_home_team_id",
    "clutch_away_team_id",
    "home_team_abbr",
    "away_team_abbr",
 
    # ── Result ──
    "home_score",
    "away_score",
    "home_winner",
    "away_winner",
    "went_to_ot",
 
    # ── Period scores ──
    "home_p1", "home_p2", "home_p3", "home_ot",
    "away_p1", "away_p2", "away_p3", "away_ot",
 
    # ── Series context ──
    "home_series_wins",
    "away_series_wins",
    "series_clinched",
    "series_summary",
    "game_number_in_series",
    "series_key",
    "round",
 
    # ── Team game stats ──
    "home_shots", "away_shots",
    "home_hits", "away_hits",
    "home_blocked_shots", "away_blocked_shots",
    "home_takeaways", "away_takeaways",
    "home_giveaways", "away_giveaways",
    "home_pp_goals", "away_pp_goals",
    "home_pp_opportunities", "away_pp_opportunities",
    "home_pp_pct", "away_pp_pct",
    "home_faceoffs_won", "away_faceoffs_won",
    "home_faceoff_pct", "away_faceoff_pct",
    "home_penalties", "away_penalties",
    "home_penalty_minutes", "away_penalty_minutes",
    "home_sh_goals", "away_sh_goals",
 
    # ── Attendance ──
    "attendance",
 
    # ── Featured athletes ──
    "winning_goalie_id", "winning_goalie_name",
    "winning_goalie_saves", "winning_goalie_sv_pct",
    "losing_goalie_id", "losing_goalie_name",
    "losing_goalie_saves", "losing_goalie_sv_pct",
    "first_star_id", "first_star_name",
    "second_star_id", "second_star_name",
    "third_star_id", "third_star_name",
 
    # ── Metadata ──
    F.lit(ingested_at).alias("ingested_at"),
    F.lit("bronze.raw_nhl_scoreboard + bronze.raw_nhl_game_summaries")
     .alias("source_tables"),
)
 
total_rows = fct_df.count()
print(f"\nTotal fctGames rows to write: {total_rows}")
 

In [0]:
# ── WRITE TO SILVER ───────────────────────────────────────────────────────────
# MERGE on source_event_id + league.
# Re-running after new round uploads adds new rows cleanly.
 
table_exists = spark.catalog.tableExists(SILVER_TABLE)
 
if not table_exists:
    (
        fct_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(SILVER_TABLE)
    )
    print(f"Table created: {SILVER_TABLE}")
 
else:
    fct_df.createOrReplaceTempView("new_fct_games")
 
    spark.sql(f"""
        MERGE INTO {SILVER_TABLE} AS target
        USING new_fct_games AS source
        ON  target.source_event_id = source.source_event_id
        AND target.league          = source.league
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"Merged into existing table: {SILVER_TABLE}")

In [0]:
# ── UPDATE dimGames.fct_event_key ─────────────────────────────────────────────
# Now that fctGames exists, populate the FK placeholder in dimGames.
# fct_event_key stores source_event_id — natural link between the two tables.
 
spark.sql(f"""
    MERGE INTO {DIM_GAMES} AS target
    USING (
        SELECT source_event_id, league
        FROM {SILVER_TABLE}
    ) AS source
    ON  target.source_event_id = source.source_event_id
    AND target.league          = source.league
    WHEN MATCHED THEN UPDATE SET
        target.fct_event_key = source.source_event_id
""")
 
# Verify update
updated = spark.sql(f"""
    SELECT COUNT(*) AS updated_rows
    FROM {DIM_GAMES}
    WHERE fct_event_key IS NOT NULL
    AND league = '{LEAGUE}'
""").collect()[0]["updated_rows"]
 
print(f"dimGames.fct_event_key updated: {updated} rows")

In [0]:
# ── VALIDATE ─────────────────────────────────────────────────────────────────
 
print("── fctGames sample ──")
spark.sql(f"""
    SELECT
        clutch_game_id,
        source_event_id,
        home_team_abbr,
        home_score,
        away_team_abbr,
        away_score,
        went_to_ot,
        home_series_wins,
        away_series_wins,
        series_clinched,
        game_number_in_series,
        home_shots,
        away_shots,
        home_hits,
        away_hits,
        home_pp_goals,
        home_pp_opportunities,
        winning_goalie_name,
        winning_goalie_saves,
        first_star_name,
        attendance
    FROM {SILVER_TABLE}
    ORDER BY round, series_key, game_number_in_series
""").show(50, truncate=False)

In [0]:
# ── SANITY CHECKS ─────────────────────────────────────────────────────────────
 
checks = spark.sql(f"""
    SELECT
        COUNT(*)                                                    AS total_games,
        COUNT(DISTINCT clutch_game_id)                             AS unique_clutch_ids,
        COUNT(CASE WHEN clutch_game_id IS NULL    THEN 1 END)      AS null_clutch_ids,
        COUNT(CASE WHEN home_winner = true        THEN 1 END)      AS home_wins,
        COUNT(CASE WHEN away_winner = true        THEN 1 END)      AS away_wins,
        COUNT(CASE WHEN went_to_ot = true         THEN 1 END)      AS ot_games,
        COUNT(CASE WHEN series_clinched = true     THEN 1 END)      AS clinching_games,
        COUNT(CASE WHEN home_shots IS NULL        THEN 1 END)      AS missing_team_stats,
        COUNT(CASE WHEN winning_goalie_id IS NULL THEN 1 END)      AS missing_goalie,
        COUNT(CASE WHEN attendance IS NULL        THEN 1 END)      AS missing_attendance,
        SUM(home_pp_goals)                                         AS total_pp_goals,
        ROUND(AVG(home_shots + away_shots), 1)                     AS avg_total_shots,
        ROUND(AVG(attendance), 0)                                  AS avg_attendance
    FROM {SILVER_TABLE}
""")
 
print("Sanity checks:")
checks.show(truncate=False)